# 🎯 Hyperparameter Tuning with Optuna + MLflow

**Project 4 of 10 — MLOps Portfolio Series**  
**Author:** Jumma Mohammad Teli | Birmingham, UK  
**Goal:** Beat Project 1 baseline AUC 0.8174 using Bayesian search

## Structure
1. Setup
2. Data Loading
3. Optuna Search
4. Convergence Analysis
5. Best Model Evaluation
6. Key Takeaways

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import mlflow
import xgboost as xgb
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from sklearn.model_selection import cross_val_score
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use('seaborn-v0_8-whitegrid')
BASELINE_AUC = 0.8174
print(f'Optuna {optuna.__version__} | Baseline AUC to beat: {BASELINE_AUC}')

In [ ]:
from src.data.loader import load_data
DATA_PATH = '../data/bank-additional-full.csv'
X_train, X_test, y_train, y_test, features = load_data(
    data_path=DATA_PATH if os.path.exists(DATA_PATH) else None
)
print(f'Train: {X_train.shape} | Test: {X_test.shape} | Positive: {y_train.mean():.1%}')

In [ ]:
trial_aucs = []
best_so_far = []

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'scale_pos_weight': trial.suggest_int('scale_pos_weight', 5, 15),
        'eval_metric': 'logloss', 'verbosity': 0, 'random_state': 42
    }
    model = xgb.XGBClassifier(**params)
    auc = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1).mean()
    trial_aucs.append(auc)
    best_so_far.append(max(trial_aucs))
    return auc

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30)
print(f'Best CV AUC: {study.best_value:.4f} | Best trial: #{study.best_trial.number}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(range(len(trial_aucs)), trial_aucs, alpha=0.6, color='#2563EB', s=30)
axes[0].plot(range(len(best_so_far)), best_so_far, color='#DC2626', lw=2, label='Best so far')
axes[0].axhline(y=BASELINE_AUC, color='orange', ls='--', lw=2, label=f'Baseline ({BASELINE_AUC})')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('CV AUC')
axes[0].set_title('Convergence Plot', fontweight='bold'); axes[0].legend()
axes[1].hist(trial_aucs, bins=15, color='#2563EB', edgecolor='white', alpha=0.8)
axes[1].axvline(x=study.best_value, color='#DC2626', lw=2, label=f'Best: {study.best_value:.4f}')
axes[1].axvline(x=BASELINE_AUC, color='orange', lw=2, ls='--', label=f'Baseline: {BASELINE_AUC}')
axes[1].set_xlabel('CV AUC'); axes[1].set_title('Trial Distribution', fontweight='bold'); axes[1].legend()
plt.suptitle('Optuna — 30 Trials XGBoost', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
best_params = {**study.best_params, 'eval_metric': 'logloss', 'verbosity': 0, 'random_state': 42}
best_model = xgb.XGBClassifier(**best_params)
best_model.fit(X_train, y_train)
y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)
test_auc = roc_auc_score(y_test, y_prob)
test_f1 = f1_score(y_test, y_pred, zero_division=0)
print(f'Test AUC: {test_auc:.4f} | Baseline: {BASELINE_AUC} | Improvement: {test_auc-BASELINE_AUC:+.4f}')
print(f'Beat baseline: {"YES" if test_auc > BASELINE_AUC else "NO"}')

In [ ]:
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

---
**Author:** Jumma Mohammad Teli | Birmingham, UK | [LinkedIn](https://linkedin.com/in/jumma-mohammad) | [GitHub](https://github.com/jumma786)

*Project 4 of 10 — MLOps Portfolio Series*